## Parameters
This notebook contains the parameters needed to customise the solution accelerator to you environment. Be sure to modify them before starting to ensure that the accelerator deploys correctly.

#### Notebook 1 & 2: Sensor data generation & Zerobus ingestion  
##### Provide required information

To use the Zerobus Ingest SDK you will need the following information:

- Your table name
  - Identify the target table you want to ingest data to. This is the table created in the notebook "1. Create-Sensor-Bronze-Table"

- Databricks Workspace URL 
  - Get your workspace URL. When viewing your Databricks workspace after logging in, take a look at the URL in your browser with the following format: https://_your_databricks-instance_.com/o=XXXXX. The URL that you require is everything before the “/o=XXXXX”

- Zerobus Host 
  - Zerobus URI = _workspace_id_.ingest.cloud.databricks.com. Databricks will provide you with this URL

- Service Principal --->  _For the following steps you need to be a workspace admin. If you are not an admin, ask to the appropriate person in your organization_
  - Go to Settings > Identity and Access.
  - Generate and save client ID and secret for that Service Principal.

In [0]:
CATALOG= "digitaltwin"
SCHEMA = "digitaltwin"
BRONZE_TABLE_NAME = "bronze_table"

# do not edit the BRONZE_TABLE definition below 
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.{BRONZE_TABLE_NAME}"

In [0]:
# the following parameters won't be required if zerobus is not used
CLIENT_ID = "your_client_ID"
CLIENT_SECRET= "your_client_secret"
ZEROBUS_URL = "your_zerobus_url"

# Auto-detect the current Databricks workspace URL (host)
try:
  ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
  WORKSPACE_URL = ctx.apiUrl().get() if ctx.apiUrl().isDefined() else "your_workspace_url"
except Exception as e:
  print(f"Warning: could not auto-detect WORKSPACE_URL: {e}")
  WORKSPACE_URL = "your_workspace_url"



#### Notebook 3: Mapping pipeline 

In [0]:
MAPPING_PIPELINE_NAME = "digitaltwin-sdp-pipeline"
TRIPLE_CATALOG = CATALOG
TRIPLE_SCHEMA = SCHEMA
TRIPLE_TABLE_NAME = "triples"

# do not edit the TRIPLE_TABLE definition 
TRIPLE_TABLE = f"{CATALOG}.{SCHEMA}.{TRIPLE_TABLE_NAME}"

#### Notebook 4 & 5: Setup Lakebase & front-end app 

In [0]:
INSTANCE_NAME = "lakebase-digitaltwin" #ensure the instance name contains only alphanumeric characters and hyphens.
SYNCED_TABLE_CATALOG = CATALOG 
SYNCED_TABLE_SCHEMA = SCHEMA
SYNCED_TABLE_NAME = "latest_sensor_triples"
SYNCED_TABLE_FULL_NAME_UC = f"{SYNCED_TABLE_CATALOG}.{SYNCED_TABLE_SCHEMA}.{SYNCED_TABLE_NAME}"
SYNCED_TABLE_FULL_NAME_PG = f"{INSTANCE_NAME}.{SCHEMA}.{SYNCED_TABLE_NAME}"  # can be dropped
TRIPLE_TABLE_FULL_NAME = TRIPLE_TABLE

# The table below will be used to store RDF definitions 
DIGITAL_TWIN_TABLE = "rdf_models"

In [0]:
APP_NAME = "digitaltwin-app"  # App name must contain only lowercase letters, numbers, and dashes

# Auto-select a SQL Warehouse:
# - Prefer the first RUNNING warehouse
# - Otherwise pick the first available warehouse
try:
  import requests

  ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
  host = ctx.apiUrl().get() if ctx.apiUrl().isDefined() else None
  token = ctx.apiToken().get() if ctx.apiToken().isDefined() else None

  WAREHOUSE_ID = None
  if host and token:
    resp = requests.get(
      f"{host}/api/2.0/sql/warehouses",
      headers={"Authorization": f"Bearer {token}"},
      timeout=30,
    )
    resp.raise_for_status()
    warehouses = resp.json().get("warehouses", [])

    running = [w for w in warehouses if w.get("state") == "RUNNING"]
    chosen = (running or warehouses)[0] if warehouses else None
    if chosen:
      WAREHOUSE_ID = chosen.get("id")

  if not WAREHOUSE_ID:
    WAREHOUSE_ID = "warehouse-id"  # fallback placeholder
except Exception as e:
  print(f"Warning: could not auto-detect SQL warehouse id: {e}")
  WAREHOUSE_ID = "warehouse-id"  # fallback placeholder
